# Issue #209 — Unified Learning Gate A: divergence-aware M×3 value model

**Contract:** BJ-UNIFIED-FULL-CYCLE-HIST-001/v3 (PI amendment 2026-08-20, accepted #238 interface).
**Audit mode:** exploration. **Scope:** Gate A development through 2023. **2024 labels stay sealed** (Gate B not opened).

Pipeline: tensor → census → distributional probability → SP quantiles → cell value → ticket/policy → bankroll → freeze identity.

Reusable package-first code under `src/issue209/`; this notebook is a thin orchestration layer.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../..")
from issue209.data import build_tensor, build_census
from issue209.models import (
    build_match_units, market_only_probabilities, fit_calibrated_logistic,
    fit_mlp_challenger, predict_logistic, predict_mlp, evaluate_calibration, _compute_losses,
)
from issue209.value import fit_sp_quantile_model, predict_sp_quantiles, cell_value_distribution
from issue209.ticket import select_ticket_panel, settle_ticket
from issue209.bankroll import simulate_bankroll, summarize_capital
import numpy as np, pandas as pd, json

## Stage 0/1 — tensor + census (REQ-209-01/02/07)

In [2]:
tensor = build_tensor()
sealed = int((tensor["issue_year"] >= 2024).sum()) + int(tensor["issue"].astype(str).str.match(r"^24").sum())
print("tensor rows:", len(tensor), "| sealed 2024 rows:", sealed, "(must be 0)")
census = build_census(tensor)
display(census)

tensor rows: 1503723 | sealed 2024 rows: 0 (must be 0)


,temporal_role,issue_year,issues,fixtures,cells,label_rate,sp_rate,n7_issues
0,calibration,2023,53,503,130806,1.0,0.3333,53
1,fit,2014,136,537,148365,1.0,0.3333,136
2,fit,2015,55,646,150687,1.0,0.3332,55
3,fit,2016,53,646,164715,1.0,0.3316,53
4,fit,2017,48,574,150615,1.0,0.3196,48
5,fit,2018,59,573,171174,1.0,0.3156,59
6,fit,2019,53,679,144981,1.0,0.3333,53
7,fit,2020,38,572,85593,1.0,0.3333,38
8,fit,2021,63,632,186405,1.0,0.3333,63
9,tune,2022,62,570,170382,1.0,0.3333,62


## Stage 2 — distributional probability learning (REQ-209-04, -03)

In [3]:
units = build_match_units(tensor)
market = market_only_probabilities(units)
logistic = fit_calibrated_logistic(units)
mlp = fit_mlp_challenger(units)
pred_l = predict_logistic(logistic, units)
pred_m = predict_mlp(mlp, units)
tune_idx = np.where(units["temporal_role"].isin(["tune"]))[0]
yt = units["realized_outcome"].astype(str).iloc[tune_idx]
tune = {
  "market_only": _compute_losses(yt, market.iloc[tune_idx])["log_loss"],
  "logistic": _compute_losses(yt, pred_l.iloc[tune_idx])["log_loss"],
  "mlp": _compute_losses(yt, pred_m.iloc[tune_idx])["log_loss"],
}
print("tune 2022 log-loss:", json.dumps(tune, indent=2))
chosen = min(tune, key=tune.get)
print("frozen candidate family:", chosen)

tune 2022 log-loss: {
  "market_only": 1.1449365368814135,
  "logistic": 1.147185290733812,
  "mlp": 1.1461928510520318
}
frozen candidate family: market_only


## Stage 3 — SP quantiles + cell-value distribution (REQ-209-04)

In [4]:
sp = fit_sp_quantile_model(units)
sp_q = predict_sp_quantiles(sp, units)
prob_chosen = market if chosen == "market_only" else (pred_m if chosen == "mlp" else pred_l)
value = cell_value_distribution(prob_chosen, sp_q, units)
print("cell value columns:", [c for c in value.columns])
display(value[["ev_raw_3","ev_raw_1","ev_raw_0"]].head())

cell value columns: ['ev_raw_3', 'ev_raw_1', 'ev_raw_0']


,ev_raw_3,ev_raw_1,ev_raw_0
0,0.369410,0.288809,0.341781
1,0.358562,0.292076,0.349363
2,0.375173,0.288850,0.335977
3,0.378882,0.289962,0.331157
4,0.360742,0.290615,0.348643


## Stage 4 — ticket/policy + bankroll on 2023 dry-run (REQ-209-05/06/08)

In [5]:
cal = units[units["temporal_role"].isin(["calibration"])].copy()
res = []
for issue in sorted(cal["issue"].unique()):
    panel = select_ticket_panel(cal, value, issue, "10h", n=7, k=8, seed=209003)
    if panel.empty:
        res.append({"issue":issue,"n_tickets":0,"stake_total":0.0,"realized_gross":0.0}); continue
    gross = 0.0
    for _, r in panel.iterrows():
        ti = tensor[tensor["issue"].astype(str)==str(issue)]
        gross += settle_ticket(r["matches"], r["outcomes"], ti)["gross_multiple"] * 2.0
    res.append({"issue":issue,"n_tickets":len(panel),"stake_total":len(panel)*2.0,"realized_gross":gross})
issue_res = pd.DataFrame(res)
path = simulate_bankroll(issue_res)
summary = summarize_capital(path)
print("2023 dry-run:", json.dumps(summary, indent=2))

2023 dry-run: {
  "ending_capital": 101087.2815835866,
  "capital_return_pct": 1.0872815835865968,
  "max_drawdown_pct": 0.25541405818829377,
  "ruin": false,
  "funded_turnover": 848.0,
  "retained_cash": 99152.0,
  "active_support_return_pct": 128.21716787577898,
  "n_issues": 53,
  "max_consecutive_losses": 16
}


## Gate A decision (REQ-209-11: 2024 sealed; Gate B not opened)

In [6]:
print("GATE_A_FROZEN_IDENTITY:", chosen)
print("Gate B (2024 terminal): NOT_OPENED — sealed until PI authorizes")

GATE_A_FROZEN_IDENTITY: market_only
Gate B (2024 terminal): NOT_OPENED — sealed until PI authorizes
